## Step 1. Importing Libraries and Configuring the Development Environment

In [4]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Disabling excessive warnings
warnings.filterwarnings("ignore")

# Configuring the display of data frames
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

# Project path configuration
BASE_DIR = Path.cwd().parent
RAW_DATA_PATH = (
    BASE_DIR / "src" / "ml_model" / "data" / "raw" / "accepted_2007_to_2018q4.csv"
)
# If the raw data is located in ml_pipeline/data/raw:
# RAW_DATA_PATH = BASE_DIR / "ml_pipeline" / "data" / "raw" / "accepted_2007_to_2018Q4.csv"


print(f"Path to the raw data: {RAW_DATA_PATH}")
print(f"The file exists: {RAW_DATA_PATH.exists()}")

Path to the raw data: D:\Projects\credit-risk-service\src\ml_model\data\raw\accepted_2007_to_2018q4.csv
The file exists: True


## Step 2. Defining the feature whitelist and loading the raw data

In [5]:
# Establishment of a strict list of features (13 features + 1 target)
FEATURE_WHITELIST = [
    # 1. Target variable
    "loan_status",
    # 2. Loan terms
    "loan_amnt",  # Loan amount ($)
    "term",  # Loan term (36 or 60 months)
    # 3. Financial profile and employment
    "annual_inc",  # Annual verified income ($)
    "dti",  # Debt-to-Income (%)
    "home_ownership",  # Type of home ownership (RENT, OWN, MORTGAGE)
    "emp_length",  # Work experience (years)
    "verification_status",  # Income verification status
    "purpose",  # Purpose of the loan
    # 4. Credit history at the time of application
    "fico_range_low",  # Minimum scoring threshold FICO
    "open_acc",  # Number of Open lines of credit
    "pub_rec",  # Number of negative public posts
    "revol_util",  # Revolving debt write-offs (%)
    "mort_acc",  # Number of mortgage accounts
]

print(f"Total number of columns to load: {len(FEATURE_WHITELIST)}")

# Reading only the relevant columns
df_raw = pd.read_csv(RAW_DATA_PATH, usecols=FEATURE_WHITELIST)

# Estimating data volume and RAM usage
ram_usage = df_raw.memory_usage(deep=True).sum() / (1024**2)
print(
    f"Dimension of the uploaded data: {df_raw.shape[0]:,} lines, {df_raw.shape[1]} columns"
)
print(f"RAM usage: {ram_usage:.2f} MB\n")

df_raw.head()

Total number of columns to load: 14
Dimension of the uploaded data: 2,260,701 lines, 14 columns
RAM usage: 370.38 MB



,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,dti,fico_range_low,open_acc,pub_rec,revol_util,mort_acc
0,3600.000,36 months,10+ years,MORTGAGE,55000.000,Not Verified,Fully Paid,debt_consolidation,5.910,675.000,7.000,0.000,29.700,1.000
1,24700.000,36 months,10+ years,MORTGAGE,65000.000,Not Verified,Fully Paid,small_business,16.060,715.000,22.000,0.000,19.200,4.000
2,20000.000,60 months,10+ years,MORTGAGE,63000.000,Not Verified,Fully Paid,home_improvement,10.780,695.000,6.000,0.000,56.200,5.000
3,35000.000,60 months,10+ years,MORTGAGE,110000.000,Source Verified,Current,debt_consolidation,17.060,785.000,13.000,0.000,11.600,1.000
4,10400.000,60 months,3 years,MORTGAGE,104433.000,Source Verified,Fully Paid,major_purchase,25.370,695.000,12.000,0.000,64.500,6.000


## Step 3. Filtering statuses and creating a target variable (is_default)

In [6]:
import gc

#
print("Breakdown of all loan statuses (in absolute values and as a percentage):")
status_counts = df_raw["loan_status"].value_counts(dropna=False)
status_pcts = df_raw["loan_status"].value_counts(dropna=False, normalize=True) * 100
print(
    pd.DataFrame({"Quantity": status_counts, "Percentage (%)": status_pcts}).to_string()
)

# 2. Defining Target Outcomes
# 0 - Reliable borrower (loan repaid in full)
# 1 - Default (loan written off as a loss / classified as in default)
target_mapping = {"Fully Paid": 0, "Charged Off": 1, "Default": 1}

# 3. Filtering the dataset and creating a column "is_default"
df_filtered = df_raw[df_raw["loan_status"].isin(target_mapping.keys())].copy()
df_filtered["is_default"] = (
    df_filtered["loan_status"].map(target_mapping).astype(np.int8)
)

# 4. Removing the original text feature "loan_status"
df_filtered.drop(columns=["loan_status"], inplace=True)

# 5. Freeing up memory from df_raw
del df_raw
gc.collect()

# 6. Checking the class balance of the target variable
print("\n" + "=" * 50)
print(f"Sample size after filtering: {len(df_filtered):,} lines")
print("Итоговое распределение целевого класса is_default:")
target_summary = pd.DataFrame(
    {
        "Quantity": df_filtered["is_default"].value_counts(),
        "Proportion (%)": df_filtered["is_default"].value_counts(normalize=True) * 100,
    }
)
target_summary.index = ["0 (Fully Paid)", "1 (Default)"]
print(target_summary.to_string())

Breakdown of all loan statuses (in absolute values and as a percentage):
                                                     Quantity  Percentage (%)
loan_status                                                                  
Fully Paid                                            1076751          47.629
Current                                                878317          38.852
Charged Off                                            268559          11.879
Late (31-120 days)                                      21467           0.950
In Grace Period                                          8436           0.373
Late (16-30 days)                                        4349           0.192
Does not meet the credit policy. Status:Fully Paid       1988           0.088
Does not meet the credit policy. Status:Charged Off       761           0.034
Default                                                    40           0.002
NaN                                                        33        

## Step 4. Feature Inspection, String Parsing, and Data Sanitization

In [7]:
# 1. Missing values audit
print("Missing values summary across selected features:")
missing_summary = pd.DataFrame(
    {
        "Missing Count": df_filtered.isnull().sum(),
        "Missing Ratio (%)": (df_filtered.isnull().sum() / len(df_filtered) * 100),
    }
)
print(missing_summary[missing_summary["Missing Count"] > 0].to_string())

# 2. Parse emp_length into numeric years (0 to 10)
emp_length_mapping = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10,
}
df_filtered["emp_length"] = df_filtered["emp_length"].map(emp_length_mapping)

# 3. Clean string artifacts in term (whitespace stripping)
df_filtered["term"] = df_filtered["term"].astype(str).str.strip()

# 4. Standardize home_ownership categories
# Merge rare edge cases (NONE, ANY) into OTHER
df_filtered["home_ownership"] = df_filtered["home_ownership"].replace(
    ["NONE", "ANY"], "OTHER"
)

# 5. Review data types adn summary statistics
print("\n" + "=" * 50)
print("Updated feature data types:")
print(df_filtered.dtypes)
print("\nSample records after sanitization:")
df_filtered.head(5)

Missing values summary across selected features:
            Missing Count  Missing Ratio (%)
emp_length          78516              5.836
dti                   374              0.028
revol_util            857              0.064
mort_acc            47281              3.514

Updated feature data types:
loan_amnt              float64
term                       str
emp_length             float64
home_ownership             str
annual_inc             float64
verification_status        str
purpose                    str
dti                    float64
fico_range_low         float64
open_acc               float64
pub_rec                float64
revol_util             float64
mort_acc               float64
is_default                int8
dtype: object

Sample records after sanitization:


,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,purpose,dti,fico_range_low,open_acc,pub_rec,revol_util,mort_acc,is_default
0,3600.000,36 months,10.000,MORTGAGE,55000.000,Not Verified,debt_consolidation,5.910,675.000,7.000,0.000,29.700,1.000,0
1,24700.000,36 months,10.000,MORTGAGE,65000.000,Not Verified,small_business,16.060,715.000,22.000,0.000,19.200,4.000,0
2,20000.000,60 months,10.000,MORTGAGE,63000.000,Not Verified,home_improvement,10.780,695.000,6.000,0.000,56.200,5.000,0
4,10400.000,60 months,3.000,MORTGAGE,104433.000,Source Verified,major_purchase,25.370,695.000,12.000,0.000,64.500,6.000,0
5,11950.000,36 months,4.000,RENT,34000.000,Source Verified,debt_consolidation,10.200,690.000,5.000,0.000,68.400,0.000,0


## Step 5. Building the unified preprocessing pipeline (ColumnTransformer)

In [8]:
# 1. Segregate features into numerical and categorical subsets
NUMERICAL_FEATURES = [
    "loan_amnt",
    "emp_length",
    "annual_inc",
    "dti",
    "fico_range_low",
    "open_acc",
    "pub_rec",
    "revol_util",
    "mort_acc",
]

CATEGORICAL_FEATURES = ["term", "home_ownership", "verification_status", "purpose"]

TARGET_COL = "is_default"

print(f"Numerical features count: {len(NUMERICAL_FEATURES)}")
print(f"Categorical count: {len(CATEGORICAL_FEATURES)}")
print(f"Total model features: {len(NUMERICAL_FEATURES) + len(CATEGORICAL_FEATURES)}")


# 2. Build dedicated preprocessing pipeline for each feature type
num_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# 3. Assemble the unified ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, NUMERICAL_FEATURES),
        ("cat", cat_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)


# 4. Fit and transform on a small to inspect output dimensions
sample_x = df_filtered[NUMERICAL_FEATURES + CATEGORICAL_FEATURES].iloc[:1000]
transformed_sample = preprocessor.fit_transform(sample_x)

# Retrieve generated one-hot feature names
encoded_cat_names = (
    preprocessor.named_transformers_["cat"]
    .named_steps["encoder"]
    .get_feature_names_out(CATEGORICAL_FEATURES)
)
all_transformed_cols = NUMERICAL_FEATURES + list(encoded_cat_names)

print("\n" + "=" * 50)
print(f"Input feature shape: {sample_x.shape}")
print(f"Transformed output shape: {transformed_sample.shape}")
print(f"Total encoded features generated: {len(all_transformed_cols)}")
print("\nGenerated one-hot categorical feature names:")
for col in encoded_cat_names:
    print(f" - {col}")

Numerical features count: 9
Categorical count: 4
Total model features: 13

Input feature shape: (1000, 13)
Transformed output shape: (1000, 28)
Total encoded features generated: 28

Generated one-hot categorical feature names:
 - term_36 months
 - term_60 months
 - home_ownership_MORTGAGE
 - home_ownership_OWN
 - home_ownership_RENT
 - verification_status_Not Verified
 - verification_status_Source Verified
 - verification_status_Verified
 - purpose_car
 - purpose_credit_card
 - purpose_debt_consolidation
 - purpose_home_improvement
 - purpose_house
 - purpose_major_purchase
 - purpose_medical
 - purpose_moving
 - purpose_other
 - purpose_small_business
 - purpose_vacation


## Step 6. Train/Test Split, Transformer Fitting, Stress-Testing, and Dataset Serialization

In [9]:
import joblib
from sklearn.model_selection import train_test_split

# 1. Split BEFORE fitting anything, so no statistic derived from the test set
# (imputation medians, scaler mean/std, one-hot categories) can leak into training.
RANDOM_STATE = 42
TEST_SIZE = 0.2

X = df_filtered[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
y = df_filtered[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train set: {X_train.shape[0]:,} rows")
print(f"Test set:  {X_test.shape[0]:,} rows")
print(
    "Train target balance (%):\n"
    f"{(y_train.value_counts(normalize=True) * 100).to_string()}"
)
print(
    "Test target balance (%):\n"
    f"{(y_test.value_counts(normalize=True) * 100).to_string()}"
)

# 2. Fit the preprocessor on the TRAIN split only
preprocessor.fit(X_train)
print("\nPreprocessor fitted on the train split only.")

# Inspect the final list of encoded features
encoded_cat_names_full = (
    preprocessor.named_transformers_["cat"]
    .named_steps["encoder"]
    .get_feature_names_out(CATEGORICAL_FEATURES)
)
total_expected_features = len(NUMERICAL_FEATURES) + len(encoded_cat_names_full)
print(
    f"Total features after fit: {total_expected_features} "
    f"({len(NUMERICAL_FEATURES)} numerical + {len(encoded_cat_names_full)} one-hot encoded)"
)

# 3. Stress-testing: adversarial applicant record
adversarial_applicant = pd.DataFrame(
    [
        {
            "loan_amnt": np.nan,
            "int_rate": np.nan,
            "installment": np.nan,
            "emp_length": np.nan,
            "annual_inc": 50000.0,
            "dti": np.nan,
            "fico_range_low": 680.0,
            "open_acc": 5.0,
            "pub_rec": 0.0,
            "revol_util": np.nan,
            "mort_acc": 1.0,
            "term": "120 months",
            "home_ownership": "SPACE_STATION",
            "verification_status": "Self-Declared",
            "purpose": "crypto_trading",
        }
    ]
)

# 4. Dynamic assertion test
try:
    stress_output = preprocessor.transform(adversarial_applicant)
    assert stress_output.shape == (
        1,
        total_expected_features,
    ), f"Expected shape (1, {total_expected_features}), got {stress_output.shape}"
    assert not np.isnan(stress_output).any(), "Transformed output contains NaN values!"
    print(
        f"Stress-test PASSED: Unseen categories ignored, shape is valid {stress_output.shape}."
    )
except Exception as e:
    print(f"Stress-test FAILED: {str(e)}")

# 5. Serialize the raw (unprocessed) train/test splits and the fitted preprocessor.
# Downstream notebooks should load these directly rather than re-splitting or
# re-fitting on the full dataset, to keep the train/test boundary consistent.
output_dir = BASE_DIR / "src" / "ml_model" / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_path = output_dir / "loans_train.parquet"
test_path = output_dir / "loans_test.parquet"
preprocessor_path = output_dir / "preprocessor.joblib"

train_df = X_train.copy()
train_df[TARGET_COL] = y_train
test_df = X_test.copy()
test_df[TARGET_COL] = y_test

train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)
joblib.dump(preprocessor, preprocessor_path)

print(
    f"\nSaved train split (Parquet): {train_path} ({train_path.stat().st_size / (1024**2):.2f} MB)"
)
print(
    f"Saved test split (Parquet):  {test_path} ({test_path.stat().st_size / (1024**2):.2f} MB)"
)
print(f"Saved fitted preprocessor:   {preprocessor_path}")

Train set: 1,076,280 rows
Test set:  269,070 rows
Train target balance (%):
is_default
0   80.035
1   19.965
Test target balance (%):
is_default
0   80.035
1   19.965

Preprocessor fitted on the train split only.
Total features after fit: 32 (9 numerical + 23 one-hot encoded)
Stress-test PASSED: Unseen categories ignored, shape is valid (1, 32).

Saved train split (Parquet): D:\Projects\credit-risk-service\src\ml_model\data\processed\loans_train.parquet (11.34 MB)
Saved test split (Parquet):  D:\Projects\credit-risk-service\src\ml_model\data\processed\loans_test.parquet (2.83 MB)
Saved fitted preprocessor:   D:\Projects\credit-risk-service\src\ml_model\data\processed\preprocessor.joblib
